# TP1 - Analyse de Données
## Guillaume Demerges - M1 DataEng

## Imports nécessaires

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('default')
sns.set_palette('husl')
%matplotlib inline

---
# Exercice 1 - Analyse des taxes d'habitation

### Question 1 : Importer le fichier capital.xls

In [ ]:
df_capital = pd.read_excel('Capital.xls')

print("Dimensions du dataset:", df_capital.shape)
print("\nPremières lignes:")
print(df_capital.head())
print("\nInformations sur les colonnes:")
print(df_capital.info())
print("\nStatistiques descriptives:")
print(df_capital.describe())

### Question 2 : Représenter graphiquement la répartition des régions (barplot et camembert)

In [ ]:
# Comptage des villes par région
region_counts = df_capital['Région'].value_counts()

# Création d'une figure avec 2 sous-graphiques
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Barplot
region_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Répartition des villes par région (Barplot)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Région', fontsize=12)
axes[0].set_ylabel('Nombre de villes', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Camembert
axes[1].pie(region_counts.values, labels=region_counts.index, autopct='%1.1f%%', 
            startangle=90, counterclock=False)
axes[1].set_title('Répartition des villes par région (Camembert)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("Nombre de villes par région:")
print(region_counts)

### Question 3 : Quelle est la région la plus représentée ?

In [ ]:
region_max = region_counts.idxmax()
nb_villes_max = region_counts.max()

print(f"La région la plus représentée est: {region_max}")
print(f"Nombre de villes: {nb_villes_max}")
print(f"Pourcentage: {(nb_villes_max / len(df_capital) * 100):.2f}%")

### Question 4 : Calculer les statistiques sur le taux de taxe d'habitation et les regrouper par région

In [ ]:
print("=== Statistiques globales sur le taux de taxe d'habitation ===")
print(df_capital['TTH'].describe())
print(f"\nMédiane: {df_capital['TTH'].median():.2f}")
print(f"Mode: {df_capital['TTH'].mode().values[0]:.2f}")

print("\n=== Statistiques par région ===")
stats_par_region = df_capital.groupby('Région')['TTH'].agg([
    ('Nombre', 'count'),
    ('Moyenne', 'mean'),
    ('Médiane', 'median'),
    ('Écart-type', 'std'),
    ('Min', 'min'),
    ('Max', 'max')
]).round(2)

stats_par_region = stats_par_region.sort_values('Moyenne')
print(stats_par_region)

# Visualisation des moyennes par région
plt.figure(figsize=(12, 6))
stats_par_region['Moyenne'].plot(kind='barh', color='coral', edgecolor='black')
plt.title('Taux moyen de taxe d\'habitation par région', fontsize=14, fontweight='bold')
plt.xlabel('Taux moyen (%)', fontsize=12)
plt.ylabel('Région', fontsize=12)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### Question 5 : Quelle est la région la plus intéressante (taux moyen le plus faible) ?

In [ ]:
region_taux_min = stats_par_region['Moyenne'].idxmin()
taux_min = stats_par_region.loc[region_taux_min, 'Moyenne']

print(f"La région la plus intéressante (taux moyen le plus faible) est: {region_taux_min}")
print(f"Taux moyen: {taux_min:.2f}%")
print(f"\nDétails pour cette région:")
print(stats_par_region.loc[region_taux_min])

### Question 6 : Quelle est la région la plus représentative de la moyenne nationale ?

In [ ]:
# Moyenne nationale
moyenne_nationale = df_capital['TTH'].mean()
print(f"Moyenne nationale: {moyenne_nationale:.2f}%")

# Calcul de l'écart entre chaque région et la moyenne nationale
stats_par_region['Écart_Moyenne_Nationale'] = abs(stats_par_region['Moyenne'] - moyenne_nationale)

# Région la plus proche de la moyenne nationale
region_representative = stats_par_region['Écart_Moyenne_Nationale'].idxmin()
ecart_min = stats_par_region.loc[region_representative, 'Écart_Moyenne_Nationale']

print(f"\nLa région la plus représentative de la moyenne nationale est: {region_representative}")
print(f"Taux moyen de la région: {stats_par_region.loc[region_representative, 'Moyenne']:.2f}%")
print(f"Écart avec la moyenne nationale: {ecart_min:.2f}%")

### Question 7 : Typologie des villes en 3 groupes selon le taux de taxe d'habitation

In [ ]:
# Création de 3 groupes en utilisant les terciles
df_capital['Groupe'] = pd.qcut(df_capital['TTH'], 
                                q=3, 
                                labels=['Faible', 'Moyen', 'Élevé'])

# Affichage des bornes des groupes
print("=== Typologie des villes en 3 groupes ===")
print("\nBornes des groupes:")
print(df_capital.groupby('Groupe')['TTH'].agg(['min', 'max', 'mean', 'count']))

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogramme avec les groupes
for groupe in ['Faible', 'Moyen', 'Élevé']:
    data = df_capital[df_capital['Groupe'] == groupe]['TTH']
    axes[0].hist(data, alpha=0.6, label=groupe, bins=10, edgecolor='black')

axes[0].set_title('Distribution du taux de taxe d\'habitation par groupe', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Taux de taxe d\'habitation (%)', fontsize=12)
axes[0].set_ylabel('Nombre de villes', fontsize=12)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Répartition des groupes
groupe_counts = df_capital['Groupe'].value_counts()
axes[1].pie(groupe_counts.values, labels=groupe_counts.index, autopct='%1.1f%%', 
            colors=['lightgreen', 'gold', 'lightcoral'])
axes[1].set_title('Répartition des villes par groupe', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Quelques exemples de villes par groupe
print("\nExemples de villes par groupe:")
for groupe in ['Faible', 'Moyen', 'Élevé']:
    print(f"\n{groupe}:")
    villes = df_capital[df_capital['Groupe'] == groupe][['Ville', 'TTH', 'Région']].head(5)
    print(villes.to_string(index=False))

### Question 8 : Villes avec un taux ±30% de la moyenne

In [ ]:
def villes_taux_extreme(df, colonne_taux, seuil_pct):
    moyenne = df[colonne_taux].mean()
    borne_sup = moyenne * (1 + seuil_pct / 100)
    borne_inf = moyenne * (1 - seuil_pct / 100)
    
    villes_au_dessus = df[df[colonne_taux] > borne_sup]
    villes_en_dessous = df[df[colonne_taux] < borne_inf]
    
    return villes_au_dessus, villes_en_dessous

villes_sup_30, villes_inf_30 = villes_taux_extreme(df_capital, 'TTH', 30)

moyenne = df_capital['TTH'].mean()
print(f"Moyenne nationale: {moyenne:.2f}%")
print(f"Borne supérieure (+30%): {moyenne * 1.3:.2f}%")
print(f"Borne inférieure (-30%): {moyenne * 0.7:.2f}%")

print(f"\n=== Villes avec un taux 30% au-dessus de la moyenne ({len(villes_sup_30)} villes) ===")
if len(villes_sup_30) > 0:
    print(villes_sup_30[['Ville', 'Région', 'TTH']].sort_values('TTH', ascending=False))
else:
    print("Aucune ville")

print(f"\n=== Villes avec un taux 30% en-dessous de la moyenne ({len(villes_inf_30)} villes) ===")
if len(villes_inf_30) > 0:
    print(villes_inf_30[['Ville', 'Région', 'TTH']].sort_values('TTH'))
else:
    print("Aucune ville")

### Question 9 : Villes avec un taux ±20% de la moyenne

In [ ]:
villes_sup_20, villes_inf_20 = villes_taux_extreme(df_capital, 'TTH', 20)

print(f"Moyenne nationale: {moyenne:.2f}%")
print(f"Borne supérieure (+20%): {moyenne * 1.2:.2f}%")
print(f"Borne inférieure (-20%): {moyenne * 0.8:.2f}%")

print(f"\n=== Villes avec un taux 20% au-dessus de la moyenne ({len(villes_sup_20)} villes) ===")
if len(villes_sup_20) > 0:
    print(villes_sup_20[['Ville', 'Région', 'TTH']].sort_values('TTH', ascending=False))
else:
    print("Aucune ville")

print(f"\n=== Villes avec un taux 20% en-dessous de la moyenne ({len(villes_inf_20)} villes) ===")
if len(villes_inf_20) > 0:
    print(villes_inf_20[['Ville', 'Région', 'TTH']].sort_values('TTH'))
else:
    print("Aucune ville")

---
# Exercice 2 - Analyse des données de pauvreté

### Question 1 : Fonction pour tracer le nuage de points avec coefficient de corrélation

In [ ]:
df_poverty = pd.read_csv('poverty.txt', sep='\s+', 
                         names=['Birth', 'Death', 'InfantDeath', 'Country', 'Continent'])

print("Dimensions du dataset:", df_poverty.shape)
print("\nPremières lignes:")
print(df_poverty.head(10))
print("\nInformations:")
print(df_poverty.info())
print("\nStatistiques:")
print(df_poverty.describe())

In [ ]:
def plot_scatter_with_correlation(df, x_col, y_col, title=None):
    # Calcul du coefficient de corrélation de Pearson
    correlation = df[x_col].corr(df[y_col])
    
    # Création du graphique
    plt.figure(figsize=(10, 6))
    plt.scatter(df[x_col], df[y_col], alpha=0.6, edgecolors='black', s=80)
    
    # Ajout d'une ligne de régression
    z = np.polyfit(df[x_col], df[y_col], 1)
    p = np.poly1d(z)
    plt.plot(df[x_col].sort_values(), p(df[x_col].sort_values()), 
             "r--", linewidth=2, label=f'Régression linéaire')
    
    if title is None:
        title = f'Nuage de points: {y_col} en fonction de {x_col}'
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel(f'{x_col} (%)', fontsize=12)
    plt.ylabel(f'{y_col} (%)', fontsize=12)
    
    # Affichage du coefficient de corrélation
    plt.text(0.05, 0.95, f'Corrélation de Pearson: r = {correlation:.4f}', 
             transform=plt.gca().transAxes, fontsize=12, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"Coefficient de corrélation entre {x_col} et {y_col}: r = {correlation:.4f}")
    
    # Interprétation
    if abs(correlation) > 0.7:
        force = "forte"
    elif abs(correlation) > 0.4:
        force = "modérée"
    else:
        force = "faible"
    
    sens = "positive" if correlation > 0 else "négative"
    print(f"Interprétation: Corrélation {force} {sens}\n")
    
    return correlation

In [ ]:
corr1 = plot_scatter_with_correlation(df_poverty, 'Birth', 'InfantDeath',
                                       'Mortalité infantile en fonction du taux de natalité')

In [ ]:
corr2 = plot_scatter_with_correlation(df_poverty, 'Death', 'InfantDeath',
                                       'Mortalité infantile en fonction du taux de mortalité')

### Question 2 : Fonction pour représenter la distribution par continent avec boxplots

In [ ]:
def plot_boxplot_by_continent(df, variable, title=None):
    # Création du graphique
    plt.figure(figsize=(12, 6))
    
    # Boxplot
    box_plot = df.boxplot(column=variable, by='Continent', 
                          figsize=(12, 6), patch_artist=True)
    
    # Personnalisation
    if title is None:
        title = f'Distribution de {variable} par continent'
    plt.suptitle('')  # Supprime le titre par défaut
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel('Continent', fontsize=12)
    plt.ylabel(f'{variable} (%)', fontsize=12)
    plt.xticks(rotation=45)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Statistiques par continent
    print(f"\n=== Statistiques de {variable} par continent ===")
    stats = df.groupby('Continent')[variable].agg([
        ('Nombre', 'count'),
        ('Moyenne', 'mean'),
        ('Médiane', 'median'),
        ('Écart-type', 'std'),
        ('Min', 'min'),
        ('Max', 'max')
    ]).round(2)
    print(stats)
    
    # Comparaison des continents
    print(f"\n=== Analyse comparative ===")
    print(f"Continent avec la moyenne la plus élevée: {stats['Moyenne'].idxmax()} ({stats['Moyenne'].max():.2f}%)")
    print(f"Continent avec la moyenne la plus faible: {stats['Moyenne'].idxmin()} ({stats['Moyenne'].min():.2f}%)")
    print(f"Continent le plus variable (écart-type): {stats['Écart-type'].idxmax()} ({stats['Écart-type'].max():.2f})")
    
    return stats

In [ ]:
stats_birth = plot_boxplot_by_continent(df_poverty, 'Birth', 
                                        'Distribution du taux de natalité par continent')

In [ ]:
stats_death = plot_boxplot_by_continent(df_poverty, 'Death',
                                        'Distribution du taux de mortalité par continent')

In [ ]:
stats_infant = plot_boxplot_by_continent(df_poverty, 'InfantDeath',
                                         'Distribution de la mortalité infantile par continent')

---
# Exercice 3 - Analyse des résultats en mathématiques des élèves

### Import et exploration des données

In [ ]:
df_students = pd.read_csv('student-mat.csv', sep=';')

print("Dimensions du dataset:", df_students.shape)
print("\nPremières lignes:")
print(df_students.head())
print("\nInformations sur les colonnes:")
print(df_students.info())
print("\nStatistiques descriptives:")
print(df_students.describe())

print(f"Statistiques sur Results:")
print(df_students['Results'].describe())

### Question 1 : Fonction de calcul du coefficient de corrélation de Pearson

In [ ]:
def pearson_correlation(x, y):
    # Conversion en numpy array
    x = np.array(x)
    y = np.array(y)
    
    # Calcul des moyennes
    mean_x = np.mean(x)
    mean_y = np.mean(y)
    
    # Calcul de la covariance
    covariance = np.sum((x - mean_x) * (y - mean_y)) / len(x)
    
    # Calcul des écarts-types
    std_x = np.sqrt(np.sum((x - mean_x)**2) / len(x))
    std_y = np.sqrt(np.sum((y - mean_y)**2) / len(y))
    
    # Calcul du coefficient de corrélation
    correlation = covariance / (std_x * std_y)
    
    return correlation

test_corr = pearson_correlation(df_students['age'], df_students['Results'])
test_corr_pandas = df_students['age'].corr(df_students['Results'])
print(f"Corrélation calculée avec notre fonction: {test_corr:.6f}")
print(f"Corrélation calculée avec pandas: {test_corr_pandas:.6f}")
print(f"Différence: {abs(test_corr - test_corr_pandas):.10f}")
print("\nLa fonction fonctionne correctement!")

### Question 2 : Fonction d'analyse des corrélations avec la variable cible

In [ ]:
def analyze_correlations_with_target(df, target_col):
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [col for col in numeric_cols if col != target_col]
    
    # Calcul des corrélations avec notre fonction
    correlations = {}
    for col in numeric_cols:
        corr = pearson_correlation(df[col], df[target_col])
        correlations[col] = corr
    
    # Création d'un DataFrame pour faciliter le traitement
    df_corr = pd.DataFrame.from_dict(correlations, orient='index', columns=['Corrélation'])
    df_corr['Corrélation_abs'] = df_corr['Corrélation'].abs()
    df_corr = df_corr.sort_values('Corrélation_abs', ascending=False)
    
    # Identification des corrélations positives et négatives
    df_corr['Type'] = df_corr['Corrélation'].apply(lambda x: 'Positive' if x > 0 else 'Négative')
    
    # Affichage du tableau
    print(f"\n=== Corrélations avec {target_col} ===")
    print(df_corr[['Corrélation', 'Type']].to_string())
    
    # Statistiques
    print(f"\n=== Statistiques ===")
    print(f"Nombre de corrélations positives: {(df_corr['Corrélation'] > 0).sum()}")
    print(f"Nombre de corrélations négatives: {(df_corr['Corrélation'] < 0).sum()}")
    print(f"\nCorrélation la plus forte (positive): {df_corr[df_corr['Corrélation'] > 0]['Corrélation'].max():.4f}")
    print(f"Variable: {df_corr[df_corr['Corrélation'] > 0]['Corrélation'].idxmax()}")
    print(f"\nCorrélation la plus forte (négative): {df_corr[df_corr['Corrélation'] < 0]['Corrélation'].min():.4f}")
    if (df_corr['Corrélation'] < 0).any():
        print(f"Variable: {df_corr[df_corr['Corrélation'] < 0]['Corrélation'].idxmin()}")
    
    # Visualisation
    plt.figure(figsize=(12, 8))
    
    # Couleurs selon le signe
    colors = ['green' if x > 0 else 'red' for x in df_corr['Corrélation']]
    
    # Création du barplot
    bars = plt.barh(range(len(df_corr)), df_corr['Corrélation'], color=colors, edgecolor='black')
    plt.yticks(range(len(df_corr)), df_corr.index)
    plt.xlabel('Coefficient de corrélation', fontsize=12)
    plt.ylabel('Variables', fontsize=12)
    plt.title(f'Corrélations avec {target_col} (triées par valeur absolue)', 
              fontsize=14, fontweight='bold')
    plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
    plt.grid(axis='x', alpha=0.3)
    
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='green', edgecolor='black', label='Corrélation positive'),
                      Patch(facecolor='red', edgecolor='black', label='Corrélation négative')]
    plt.legend(handles=legend_elements, loc='best')
    
    plt.tight_layout()
    plt.show()
    
    return df_corr[['Corrélation', 'Type']]

In [ ]:
correlations_results = analyze_correlations_with_target(df_students, 'Results')

### Question 3 : Les 3 variables les plus corrélées avec Results

In [ ]:
top_3 = correlations_results.nlargest(3, correlations_results.columns[0])

print("=== Les 3 variables les plus corrélées avec Results ===")
print(top_3)

print("\n=== Analyse détaillée ===")
for i, (var, row) in enumerate(top_3.iterrows(), 1):
    print(f"\n{i}. {var}")
    print(f"   Corrélation: {row['Corrélation']:.4f}")
    print(f"   Type: {row['Type']}")
    
    # Interprétation de la force
    abs_corr = abs(row['Corrélation'])
    if abs_corr > 0.7:
        force = "très forte"
    elif abs_corr > 0.5:
        force = "forte"
    elif abs_corr > 0.3:
        force = "modérée"
    else:
        force = "faible"
    print(f"   Interprétation: Corrélation {force}")

In [ ]:
# Visualisation des relations entre Results et les 3 variables les plus corrélées
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (var, row) in enumerate(top_3.iterrows()):
    axes[idx].scatter(df_students[var], df_students['Results'], alpha=0.5, edgecolors='black')
    
    # Ligne de régression
    z = np.polyfit(df_students[var], df_students['Results'], 1)
    p = np.poly1d(z)
    axes[idx].plot(df_students[var].sort_values(), p(df_students[var].sort_values()), 
                   "r--", linewidth=2)
    
    axes[idx].set_xlabel(var, fontsize=11)
    axes[idx].set_ylabel('Results', fontsize=11)
    axes[idx].set_title(f'{var} vs Results\nr = {row["Corrélation"]:.4f}', 
                       fontsize=12, fontweight='bold')
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()